In [ ]:
from IPython.display import Image, display
display(Image(filename='/kaggle/input/datasets/mhassansaboor/psl-thumbnail/psl.png'))

# **🏏 HBL PSL 2026 Cricket Players Auction Analysis**

Welcome to a **comprehensive Exploratory Data Analysis (EDA)** of the **HBL PSL 2026 Player Auction Prices**.  
This notebook aims to uncover patterns, trends, and insights from the auction data, helping us understand **how teams spent their budgets**, which players were **most valued**, and how the **auction dynamics shaped the squads**.

---

## **🔹 Notebook Highlights**

- 📊 **Overall Auction Price Landscape**  
  Analyze the distribution of player prices, check for skewness, and detect **top-heavy spending patterns**.

- 💰 **Highest-Paid Players**  
  Identify the top 5 most expensive players, their price comparisons, and percentage gaps between ranks.

- 🏅 **Price Tier Segmentation**  
  Classify players into **Elite, Premium, Mid-tier, and Emerging** categories and analyze tier-wise spending.

- 🌏 **Local vs Foreign Players Spend**  
  Compare **average price, total spending, and top earners** for local Pakistani vs foreign players.

- 📈 **Price Ranking & Player Value Gaps**  
  Examine **rank differences, steep drop zones**, and largest price jumps between consecutive players.

- 🔍 **Correlation Patterns & Price Decay**  
  Investigate **rank-price correlation** and the exponential decay of player prices in the auction.

- 📝 **Price Conversion & Formatting Accuracy**  
  Validate Urdu price formatting against numeric estimates and standardize the data for consistency.

---

## **🎯 Objectives**

1. Provide a **clear understanding** of the PSL 2026 auction spending trends.  
2. Highlight **key players, top spending tiers, and rank-value gaps**.  
3. Present insights in a **visually appealing, PSL-themed style** suitable for reports and portfolio showcases.  

Let's dive in and explore the **PSL 2026 auction prices**! 🚀

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("/kaggle/input/hbl-psl-2026-cricket-players-auction-prices/list of cricket players and prices for HBL PSL 2026.csv")

df.info()

In [ ]:
df.head(10)

In [ ]:
plt.style.use("default")

psl_colors = {
    "green": "#006600",        # PSL Official Green
    "gold": "#D4AF37",         # Trophy Gold
    "cream": "#F2EFE5",        # Light Background
    "black": "#0A0A0A",        # Dark Base
    "blue": "#1A73E8"          # Accent
}

# Apply global Matplotlib theme
plt.rcParams.update({
    "figure.facecolor": psl_colors["cream"],
    "axes.facecolor": psl_colors["cream"],
    "axes.edgecolor": psl_colors["black"],
    "axes.labelcolor": psl_colors["black"],
    "xtick.color": psl_colors["black"],
    "ytick.color": psl_colors["black"],
    "text.color": psl_colors["black"],
    "axes.titleweight": "bold",
    "axes.titlecolor": psl_colors["green"],
    "grid.color": psl_colors["gold"],
    "grid.linestyle": "--",
    "grid.alpha": 0.3,
    "axes.prop_cycle": plt.cycler(color=[
        psl_colors["green"],
        psl_colors["gold"],
        psl_colors["blue"],
        psl_colors["black"]
    ])
})

# Seaborn Theme
sns.set_theme(
    style="whitegrid",
    rc={
        "axes.facecolor": psl_colors["cream"],
        "figure.facecolor": psl_colors["cream"],
        "grid.color": psl_colors["gold"],
        "axes.labelcolor": psl_colors["black"],
        "xtick.color": psl_colors["black"],
        "ytick.color": psl_colors["black"],
        "text.color": psl_colors["black"],
        "axes.titlecolor": psl_colors["green"]
    }
)

print("PSL theme applied successfully!")

# **🔹 1. Overall Auction Price Landscape**

In [ ]:
df["Price"] = df["Price (Numeric Estimate)"].str.replace(",", "").astype(int)

## **📌 Basic Statistics**

In [ ]:
mean_price = df["Price"].mean()
median_price = df["Price"].median()
variance_price = df["Price"].var()
skewness = df["Price"].skew()

print("Mean Price: ", mean_price)
print("Median Price: ", median_price)
print("Variance: ", variance_price)
print("Skewness: ", skewness)

## **📊 Auction Price Distribution**

This plot will show if the distribution is right-skewed.

In [ ]:
plt.figure(figsize=(10,5))
sns.histplot(df["Price"], kde=True, bins=10)
plt.title("PSL 2026 Auction Price Distribution")
plt.xlabel("Player Price (PKR)")
plt.ylabel("Frequency")
plt.grid(True)
plt.show()

## **📊 Detecting Top-Heavy Spending Pattern**

Players above the upper whisker indicate very expensive outliers → `Top-Heavy Auction`.

In [ ]:
plt.figure(figsize=(10,4))
sns.boxplot(x=df["Price"])
plt.title("PSL 2026 Auction Price Spread (Boxplot)")
plt.xlabel("Player Price (PKR)")
plt.grid(True)
plt.show()

## **📊 Smooth Distribution Curve**

Great for visualizing skewness.

In [ ]:
plt.figure(figsize=(10,5))
sns.kdeplot(df["Price"], fill=True, linewidth=2)
plt.title("Price Density Curve – Checking for Right Skew")
plt.xlabel("Player Price")
plt.grid(True)
plt.show()

## **📊 Ranked Prices**

If bars drop sharply → spending is top-heavy.

In [ ]:
df_sorted = df.sort_values("Price", ascending=False)

plt.figure(figsize=(12,6))
sns.barplot(
    data=df_sorted,
    x="Player Name (English)",
    y="Price"
)
plt.xticks(rotation=75)
plt.title("PSL 2026 Auction – Player Prices (High to Low)")
plt.xlabel("Player")
plt.ylabel("Price (PKR)")
plt.grid(True)
plt.show()

## **📊 Price Drop-Off Curve**

A very professional plot showing spending pattern.

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(df_sorted["Price"], marker="o", linewidth=2)
plt.title("Price Decay Curve – Detecting Top-Heavy Spending")
plt.xlabel("Ranked Players (Highest to Lowest Price)")
plt.ylabel("Price (PKR)")
plt.grid(True)
plt.show()

# **🔹 2. Highest-Paid Players**

## **🔹Extract Top 5 Players**

In [ ]:
df_sorted = df.sort_values("Price", ascending=False)
top5 = df_sorted.head(5)
top5

## **🔹 Percentage Difference Between Rank 1 and Rank 5**

In [ ]:
price_1 = top5.iloc[0]["Price"]
price_5 = top5.iloc[-1]["Price"]

pct_diff = ((price_1 - price_5) / price_1) * 100

print("Highest Paid Player Price:", price_1)
print("5th Highest Player Price:", price_5)
print("Percentage Difference:", round(pct_diff, 2), "%")

## **📊 Top 5 Players (Bar Chart with English Names)**

In [ ]:
plt.figure(figsize=(10,6))
sns.barplot(
    data=top5,
    x="Player Name (English)",
    y="Price"
)
plt.title("Top 5 Highest-Paid Players – English Names")
plt.xlabel("Player (English)")
plt.ylabel("Price (PKR)")
plt.xticks(rotation=45)
plt.grid(True)
plt.show()

## **📊 Top 5 Players (Bar Chart with Urdu Names)**

In [ ]:
plt.figure(figsize=(10,6))
sns.barplot(
    data=top5,
    x="Player Name (Urdu)",
    y="Price"
)
plt.title("Top 5 Highest-Paid Players – Urdu Names")
plt.xlabel("کھلاڑی (اردو)")
plt.ylabel("قیمت (PKR)")
plt.xticks(rotation=45)
plt.grid(True)
plt.show()

## **📊 Side-by-Side Comparison (English + Numeric)**

In [ ]:
plt.figure(figsize=(10,6))
sns.barplot(
    data=top5,
    x="Player Name (English)",
    y="Price",
)
for i, v in enumerate(top5["Price"]):
    plt.text(i, v + (v*0.015), f"{v:,}", ha='center', fontsize=10)

plt.title("Top 5 Players – Price Comparison (PKR)")
plt.xlabel("Player")
plt.ylabel("Price (PKR)")
plt.xticks(rotation=45)
plt.grid(True)
plt.show()

## **📊 Price Gap Line Plot (Detecting Differences)**

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(top5["Player Name (English)"], top5["Price"], marker="o", linewidth=2)
plt.title("Price Drop-Off Among Top 5 Highest-Paid Players")
plt.xlabel("Players")
plt.ylabel("Price (PKR)")
plt.grid(True)
plt.show()

# **🔹 3. Price Tier Segmentation**

## **🔹 Create Salary Tier Categories**

In [ ]:
# Define function for salary tier classification
def price_tier(price):
    if price >= 50_000_000:
        return "Elite (50M+)"
    elif 30_000_000 <= price < 50_000_000:
        return "Premium (30M–50M)"
    elif 20_000_000 <= price < 30_000_000:
        return "Mid-Tier (20M–30M)"
    else:
        return "Emerging (<20M)"

# Apply classification
df["Tier"] = df["Price"].apply(price_tier)

df.head()

## **🔹 Count Players in Each Category**

In [ ]:
tier_counts = df["Tier"].value_counts()
tier_counts

## **🔹 Average Price per Category**

In [ ]:
tier_avg = df.groupby("Tier")["Price"].mean().sort_values(ascending=False)
tier_avg

## **📊 Number of Players in Each Tier**

In [ ]:
plt.figure(figsize=(10,6))
sns.countplot(
    data=df,
    x="Tier",
    order=["Elite (50M+)", "Premium (30M–50M)", "Mid-Tier (20M–30M)", "Emerging (<20M)"]
)
plt.title("Number of Players in Each Price Tier")
plt.xlabel("Price Tier")
plt.ylabel("Count")
plt.grid(True)
plt.show()

## **📊 Average Price per Tier**

In [ ]:
plt.figure(figsize=(10,6))
sns.barplot(
    x=tier_avg.index,
    y=tier_avg.values
)
plt.title("Average Price per Tier")
plt.xlabel("Tier")
plt.ylabel("Average Price (PKR)")
plt.xticks(rotation=20)
plt.grid(True)
plt.show()

## **📊 Tier Distribution**

In [ ]:
plt.figure(figsize=(8,8))
plt.pie(
    tier_counts.values,
    labels=tier_counts.index,
    autopct="%1.1f%%",
    startangle=90
)
plt.title("PSL 2026 — Tier Distribution of Players")
plt.show()

## **📊 Foreign vs Local in Each Tier**

In [ ]:
# List of known foreign players (you can add/remove)
foreign_players = [
    "Daryl Mitchell", "David Warner", "Rilee Rossouw",
    "Kusal Perera", "Rishad Hossain", "Max Bryant"
]

# Make sure strings match by stripping spaces
df["Player Name (English)"] = df["Player Name (English)"].str.strip()

# Add nationality column safely
df["Nationality"] = df["Player Name (English)"].apply(
    lambda x: "Foreign" if x in foreign_players else "Local"
)

df["Nationality"].value_counts()


plt.figure(figsize=(12,6))
sns.countplot(
    data=df,
    x="Tier",
    hue="Nationality",
    order=["Elite (50M+)", "Premium (30M–50M)", "Mid-Tier (20M–30M)", "Emerging (<20M)"]
)
plt.title("Local vs Foreign Players by Price Tier")
plt.xlabel("Price Tier")
plt.ylabel("Number of Players")
plt.grid(True)
plt.show()

## **📊 Boxplot of Price by Tier**

In [ ]:
plt.figure(figsize=(12,6))
sns.boxplot(
    data=df,
    x="Tier",
    y="Price",
    order=["Elite (50M+)", "Premium (30M–50M)", "Mid-Tier (20M–30M)", "Emerging (<20M)"]
)
plt.title("Price Distribution by Tier")
plt.xlabel("Tier")
plt.ylabel("Price (PKR)")
plt.grid(True)
plt.show()

# **🔹 4. Foreign Players vs Local Players Spend**

## **📌 Tag Local vs Foreign Players**

In [ ]:
df["PlayerType"] = df["Nationality"].apply(
    lambda x: "Local (Pakistan)" if x == "Pakistan" else "Foreign"
)

## **✅ Clean Price `(Numeric Estimate)`**

In [ ]:
# Remove commas and convert to integer
df["Price (Numeric Estimate)"] = df["Price (Numeric Estimate)"].str.replace(",", "")
df["Price (Numeric Estimate)"] = df["Price (Numeric Estimate)"].astype(int)

# Check
df[["Player Name (English)", "Price (Numeric Estimate)"]].head()

## **📊 Average Price: Local vs Foreign**

In [ ]:
avg_price = df.groupby("PlayerType")["Price (Numeric Estimate)"].mean().reset_index()

plt.figure(figsize=(7,5))
sns.barplot(
    data=avg_price,
    x="PlayerType",
    y="Price (Numeric Estimate)",
    palette=["#006600", "#d4af37"]  # Green = Local, Gold = Foreign
)

plt.title("📌 Average Price: Local vs Foreign Players (PKR)")
plt.xlabel("")
plt.ylabel("Average Price (PKR)")
plt.grid(True, linestyle="--", alpha=0.3)
plt.show()

## **📊 Total Spending on Local vs Foreign Players**

In [ ]:
total_spend = df.groupby("PlayerType")["Price (Numeric Estimate)"].sum().reset_index()

plt.figure(figsize=(7,5))
sns.barplot(
    data=total_spend,
    x="PlayerType",
    y="Price (Numeric Estimate)",   # Correct column name
    palette=["#66cc66", "#d4af37"]  # Green = Local, Gold = Foreign
)

plt.title("💰 Total Spending: Local vs Foreign Players (PKR)")
plt.xlabel("")
plt.ylabel("Total Spend (PKR)")
plt.grid(True, linestyle="--", alpha=0.3)
plt.show()

## **📊 Percentage Spend Breakdown**

In [ ]:
plt.figure(figsize=(6,6))
plt.pie(
    total_spend["Price (Numeric Estimate)"],
    labels=total_spend["PlayerType"],
    autopct="%1.1f%%",
    colors=["#006600", "#d4af37"],
    textprops={"color": "white"}
)
plt.title("🔘 Spending % on Local vs Foreign Talent")
plt.show()

## **📊 Highest-Paid Local vs Foreign Player**

In [ ]:
highest_local = df[df["PlayerType"]=="Local (Pakistan)"].sort_values(
    "Price (Numeric Estimate)", ascending=False
).head(1)

highest_foreign = df[df["PlayerType"]=="Foreign"].sort_values(
    "Price (Numeric Estimate)", ascending=False
).head(1)

top_two = pd.concat([highest_local, highest_foreign])

plt.figure(figsize=(7,5))
sns.barplot(
    data=top_two,
    x="Player Name (English)",              # Correct column
    y="Price (Numeric Estimate)",
    palette=["#66cc66", "#d4af37"]          # Green = Local, Gold = Foreign
)

plt.title("⭐ Highest-Paid Local vs Foreign Players (PKR)")
plt.xlabel("")
plt.ylabel("Price (PKR)")
plt.grid(True, linestyle="--", alpha=0.3)
plt.show()

## **📊 Count of Players: Local vs Foreign**

In [ ]:
count_players = df["PlayerType"].value_counts().reset_index()
count_players.columns = ["PlayerType", "Count"]

plt.figure(figsize=(7,5))
sns.barplot(
    data=count_players,
    x="PlayerType",
    y="Count",
    palette=["#006600", "#d4af37"]
)
plt.title("👥 Number of Players: Local vs Foreign")
plt.xlabel("")
plt.ylabel("Count")
plt.show()

# **🔹 5. Price Ranking & Player Value Gaps**

## **✅ Sort Players by Price**

In [ ]:
df["Price (Numeric Estimate)"] = df["Price (Numeric Estimate)"].astype(int)
df_sorted = df.sort_values("Price (Numeric Estimate)", ascending=False).reset_index(drop=True)
df_sorted["Rank"] = df_sorted.index + 1
df_sorted.head(10)

## **✅ Calculate Price Gaps Between Consecutive Players**

In [ ]:
df_sorted["Price_Gap"] = df_sorted["Price (Numeric Estimate)"].diff(periods=-1).abs()  
df_sorted[["Rank", "Player Name (English)", "Price (Numeric Estimate)", "Price_Gap"]].head(10)

## **✅ Prices (Highlight Drop Zones)**

In [ ]:
plt.figure(figsize=(12,6))
sns.lineplot(
    data=df_sorted,
    x="Rank",
    y="Price (Numeric Estimate)",
    marker="o",
    color="#d4af37"
)
plt.fill_between(
    df_sorted["Rank"],
    df_sorted["Price (Numeric Estimate)"],
    color="#006600",
    alpha=0.1
)
plt.title("📈 PSL 2026 Player Price Ranking & Drop Zones")
plt.xlabel("Player Rank")
plt.ylabel("Price (PKR)")
plt.grid(True, linestyle="--", alpha=0.3)
plt.show()

## **✅ Price Gaps Between Consecutive Players**

In [ ]:
plt.figure(figsize=(12,6))
sns.barplot(
    data=df_sorted,
    x="Rank",
    y="Price_Gap",
    palette="YlGn"  # Yellow-Green gradient
)
plt.title("💸 Price Gaps Between Consecutive Players")
plt.xlabel("Player Rank")
plt.ylabel("Price Difference with Next Player (PKR)")
plt.grid(True, linestyle="--", alpha=0.3)
plt.show()

## **✅ Largest Price Gap**

In [ ]:
largest_gap = df_sorted.loc[df_sorted["Price_Gap"].idxmax()]
print(f"Largest price jump: Rank {largest_gap['Rank']} - {largest_gap['Player Name (English)']} → Gap: {largest_gap['Price_Gap']:,} PKR")

## **🔹 Top 5 Drop Zones on Plot**

In [ ]:
top_gaps = df_sorted.nlargest(5, "Price_Gap")

plt.figure(figsize=(12,6))
sns.lineplot(data=df_sorted, x="Rank", y="Price (Numeric Estimate)", marker="o", color="#d4af37")
plt.scatter(top_gaps["Rank"], top_gaps["Price (Numeric Estimate)"], color="#ff0000", s=100, label="Top Gaps")
plt.title("📈 PSL 2026 Price Ranking with Top Drop Zones")
plt.xlabel("Player Rank")
plt.ylabel("Price (PKR)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.3)
plt.show()

# **🔹 6. Correlation Patterns**

## **✅ Prepare Data**

In [ ]:
# Ensure numeric columns
df_sorted["Price (Numeric Estimate)"] = df_sorted["Price (Numeric Estimate)"].astype(int)
df_sorted["Rank"] = df_sorted.index + 1

## **✅ Correlation Between Rank & Price**

In [ ]:
corr = df_sorted[["Rank", "Price (Numeric Estimate)"]].corr().iloc[0,1]
print(f"Correlation between Rank and Price: {corr:.2f}")

## **✅ Price Decay**

In [ ]:
plt.figure(figsize=(12,6))
sns.scatterplot(
    data=df_sorted,
    x="Rank",
    y="Price (Numeric Estimate)",
    s=100,
    color="#d4af37"
)
sns.regplot(
    data=df_sorted,
    x="Rank",
    y="Price (Numeric Estimate)",
    scatter=False,
    color="#006600",
    line_kws={"linewidth":2, "label":"Trendline"}
)
plt.title("📉 PSL 2026 Player Price vs Rank (Price Decay Pattern)")
plt.xlabel("Player Rank")
plt.ylabel("Price (PKR)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.3)
plt.show()

## **✅ Log-Scale to Emphasize Decay**

In [ ]:
plt.figure(figsize=(12,6))
sns.scatterplot(
    data=df_sorted,
    x="Rank",
    y="Price (Numeric Estimate)",
    s=100,
    color="#d4af37"
)
sns.lineplot(
    data=df_sorted,
    x="Rank",
    y=np.log10(df_sorted["Price (Numeric Estimate)"]),
    color="#006600",
    linewidth=2,
    label="Log Price Trend"
)
plt.title("📉 PSL 2026 Price Decay (Log Scale)")
plt.xlabel("Player Rank")
plt.ylabel("Log10(Price)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.3)
plt.show()

# **🏁 Conclusion & Key Takeaways**

We have successfully explored the **HBL PSL 2026 Player Auction Prices** and uncovered several interesting insights:

- 💸 **Top-heavy spending:** The auction clearly favored a few elite players, capturing the majority of the budget.  
- 🏅 **Highest-paid players:** Top 3–5 players dominated spending, with significant price gaps to the rest of the squad.  
- 📊 **Tier segmentation:** Most players fall in the **Premium and Mid-tier** categories, while only a few are Elite.  
- 🌏 **Local vs Foreign:** Spending on foreign talent is slightly higher per player, but local players form the majority of the squads.  
- 📉 **Price ranking & decay:** Prices drop steeply after top-ranked players, showing an exponential decay pattern in auction values.  
- 📝 **Data consistency:** Urdu price texts align well with numeric estimates, ensuring high-quality, clean data.

---

## **🎯 Final Thoughts**

This EDA highlights how **team strategies, player value, and spending patterns** shaped the PSL 2026 auction.  
It also demonstrates **how detailed visualization and structured analysis** can turn raw auction data into actionable insights for fans, analysts, and decision-makers.  

---

Thank you for exploring this analysis! 🙌  
Stay tuned for more **PSL data insights and visualizations**. 🚀